# 05 · Hierarchical summarisation

```
meeting → chunks → per-chunk summaries → combined → final executive summary
```

- Model: `sshleifer/distilbart-cnn-12-6` (free; auto-downgraded policy; `facebook/bart-large-cnn` optional).
- Safeguards: empty-chunk skip, input clamped at sentence boundaries, `no_repeat_ngram_size`,
  second-level recursion when combined summaries exceed the encoder budget, OOM → CPU retry.
- Outputs: `outputs/summaries/<meeting>_summary.json` with `chunk_summaries`,
  `combined_summary`, `final_summary`, `key_points`.

In [ ]:
print('[1/4] Load processed chunks')
import sys; sys.path.insert(0, '.')
from pathlib import Path
from src.config import load_config, resolve_path
from src.chunking import Chunk
from src.utils import load_json
cfg = load_config()
proc = resolve_path(cfg, 'data_processed')
files = sorted(proc.glob('*_chunks.json'))
if not files:
    print('No processed chunks yet -> run: !python scripts/preprocess_data.py')
chunks = [Chunk(**c) for c in load_json(files[0])] if files else []
print(files[0].name if files else '-', '| chunks:', len(chunks))

In [ ]:
print('[2/4] Run hierarchical summarisation (downloads ~1.2 GB on first run)')
from src.summarizer import HierarchicalSummarizer
from src.utils import select_models
sel = select_models(cfg)
s = HierarchicalSummarizer(model_name=sel['summarization'], device=sel['device'],
                           chunk_summary_max=cfg.summarization.chunk_summary_max_tokens,
                           chunk_summary_min=cfg.summarization.chunk_summary_min_tokens,
                           final_summary_max=cfg.summarization.final_summary_max_tokens,
                           final_summary_min=cfg.summarization.final_summary_min_tokens)
bundle = s.summarize_meeting(chunks, meeting_id=files[0].stem.replace('_chunks', ''))
print('chunk summaries:', len(bundle['chunk_summaries']))

In [ ]:
print('[3/4] Inspect: chunk summaries -> combined -> final')
for r in bundle['chunk_summaries'][:4]:
    print(f"  {r['chunk_id']}: {r['summary'][:110]}")
print()
print('COMBINED:\n', bundle['combined_summary'][:600])
print()
print('FINAL:\n', bundle['final_summary'])

In [ ]:
print('[4/4] Key discussion points')
for i, p in enumerate(bundle['key_points'], 1):
    print(f'  {i}. {p}')
if not bundle['key_points']:
    print('  Not specified')